In [6]:
import pandas as pd
import numpy as np

ratings = pd.read_csv(r"D:\Data Science\Projects\Movies_Recommendation\ratings.csv")
movies = pd.read_csv(r"D:\Data Science\Projects\Movies_Recommendation\movies.csv")
tags = pd.read_csv(r"D:\Data Science\Projects\Movies_Recommendation\tags.csv")


movies.head()

,movieId,title,genres
0,1,Toy Story (1995),Adventure|Animation|Children|Comedy|Fantasy
1,2,Jumanji (1995),Adventure|Children|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama|Romance
4,5,Father of the Bride Part II (1995),Comedy


In [7]:
data = pd.merge(ratings, movies, on="movieId")
data.head()

,userId,movieId,rating,timestamp,title,genres
0,1,31,2.5,1260759144,Dangerous Minds (1995),Drama
1,1,1029,3.0,1260759179,Dumbo (1941),Animation|Children|Drama|Musical
2,1,1061,3.0,1260759182,Sleepers (1996),Thriller
3,1,1129,2.0,1260759185,Escape from New York (1981),Action|Adventure|Sci-Fi|Thriller
4,1,1172,4.0,1260759205,Cinema Paradiso (Nuovo cinema Paradiso) (1989),Drama


In [8]:
user_movie_matrix = data.pivot_table(index='userId', columns='title', values='rating')

user_movie_matrix.head()

title,"""Great Performances"" Cats (1998)",$9.99 (2008),'Hellboy': The Seeds of Creation (2004),'Neath the Arizona Skies (1934),'Round Midnight (1986),'Salem's Lot (2004),'Til There Was You (1997),"'burbs, The (1989)",'night Mother (1986),(500) Days of Summer (2009),...,Zulu (1964),Zulu (2013),[REC] (2007),eXistenZ (1999),loudQUIETloud: A Film About the Pixies (2006),xXx (2002),xXx: State of the Union (2005),¡Three Amigos! (1986),À nous la liberté (Freedom for Us) (1931),İtirazım Var (2014)
userId,,,,,,,,,,,,,,,,,,,,,
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
user_movie_matrix_filled = user_movie_matrix.fillna(0)

In [10]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(user_movie_matrix_filled)
user_similarity_df = pd.DataFrame(user_similarity, index=user_movie_matrix_filled.index, columns=user_movie_matrix_filled.index)

user_similarity_df.head()

userId,1,2,3,4,5,6,7,8,9,10,...,662,663,664,665,666,667,668,669,670,671
userId,,,,,,,,,,,,,,,,,,,,,
1,1.000000,0.000000,0.000000,0.074482,0.016818,0.000000,0.083884,0.000000,0.012843,0.000000,...,0.000000,0.000000,0.014481,0.043719,0.000000,0.000000,0.000000,0.062917,0.000000,0.017466
2,0.000000,1.000000,0.124295,0.118821,0.103646,0.000000,0.212985,0.113190,0.113333,0.043213,...,0.477306,0.063202,0.077784,0.164162,0.466281,0.425462,0.084646,0.024140,0.170595,0.113175
3,0.000000,0.124295,1.000000,0.081640,0.151531,0.060691,0.154714,0.249781,0.134475,0.114672,...,0.161205,0.064198,0.176222,0.158357,0.177098,0.124562,0.124911,0.080984,0.136606,0.170193
4,0.074482,0.118821,0.081640,1.000000,0.130649,0.079648,0.319745,0.191013,0.030417,0.137186,...,0.114319,0.047228,0.136647,0.254030,0.121905,0.088735,0.068483,0.104309,0.054512,0.211609
5,0.016818,0.103646,0.151531,0.130649,1.000000,0.063796,0.095888,0.165712,0.086616,0.032370,...,0.191029,0.021142,0.146246,0.224245,0.139721,0.058252,0.042926,0.038358,0.062642,0.225086


In [18]:
def recommend_movies(user_id, n_recommendations=5):
    # Get similarity scores for the user
    sim_scores = user_similarity_df[user_id]
    
    # Weighted ratings of all movies
    weighted_ratings = user_movie_matrix_filled.T.dot(sim_scores) / sim_scores.sum()
    
    # Remove movies already rated by the user
    already_rated = user_movie_matrix_filled.loc[user_id]
    weighted_ratings = weighted_ratings[already_rated.isna() | (already_rated==0)]
    
    # Sort top N recommendations
    top_movies = weighted_ratings.sort_values(ascending=False).head(n_recommendations)
    return top_movies

# Example: Recommend 5 movies for user 1
print(recommend_movies(1, 5))

title
Star Wars: Episode IV - A New Hope (1977)                                         2.274263
Pulp Fiction (1994)                                                               2.201521
Star Wars: Episode V - The Empire Strikes Back (1980)                             2.141632
Shawshank Redemption, The (1994)                                                  2.099555
Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)    2.003205
dtype: float64


In [19]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

In [24]:
train_matrix = train_data.pivot_table(index='userId', columns='movieId', values='rating')
train_matrix = train_matrix.fillna(0)

In [25]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

user_similarity = cosine_similarity(train_matrix)

In [26]:
similarity_sum = np.abs(user_similarity).sum(axis=1).reshape(-1, 1)

predicted_ratings = user_similarity.dot(train_matrix) / similarity_sum

In [27]:
from sklearn.metrics import mean_squared_error

actual = []
predicted = []

for _, row in test_data.iterrows():
    user = row['userId']
    movie = row['movieId']

    if user in train_matrix.index and movie in train_matrix.columns:
        user_index = train_matrix.index.get_loc(user)
        movie_index = train_matrix.columns.get_loc(movie)

        actual.append(row['rating'])
        predicted.append(predicted_ratings[user_index, movie_index])


rmse = np.sqrt(mean_squared_error(actual, predicted))
print("RMSE:", rmse)

RMSE: 3.2895717966982363


In [28]:
from sklearn.model_selection import train_test_split
train_data, test_data = train_test_split(ratings, test_size=0.2, random_state=42)

In [30]:
import numpy as np

train_matrix = train_data.pivot_table(index='userId', columns='movieId', values='rating')

In [31]:
user_means = train_matrix.mean(axis=1)

train_centered = train_matrix.sub(user_means, axis=0).fillna(0)

In [33]:
from sklearn.metrics.pairwise import cosine_similarity

user_similarity = cosine_similarity(train_centered)

In [40]:
similarity_sum = np.abs(user_similarity).sum(axis=1).reshape(-1, 1)

# Avoid division by zero
similarity_sum[similarity_sum == 0] = 1
predicted_centered = user_similarity.dot(train_centered) / similarity_sum

# Add back user mean
predicted_ratings = predicted_centered + user_means.values.reshape(-1, 1)

In [41]:
from sklearn.metrics import mean_squared_error

actual = []
predicted = []

for _, row in test_data.iterrows():
    user = row['userId']
    movie = row['movieId']

    if user in train_matrix.index and movie in train_matrix.columns:
        user_index = train_matrix.index.get_loc(user)
        movie_index = train_matrix.columns.get_loc(movie)

        actual.append(row['rating'])
        predicted.append(predicted_ratings[user_index, movie_index])

rmse = np.sqrt(mean_squared_error(actual, predicted))
print("RMSE:", rmse)


RMSE: 0.9424425789835968
